## Задача 1. My heart will go on

Датасет **titanic** из библиотеки `Seaborn` содержит информацию о пассажирах легендарного корабля Титаник, который затонул в 1912 году после столкновения с айсбергом. Этот набор данных часто используется для обучения и тестирования алгоритмов машинного обучения, особенно в задачах бинарной классификации (выжил / не выжил).

**Описание данных**

| Поле         | Тип      | Описание |
|--------------|----------|----------|
| `survived`   | int      | Выжил (1) или не выжил (0) |
| `pclass`     | int      | Класс билета (1, 2, 3) |
| `sex`        | str      | Пол (`male`/`female`) |
| `age`        | float    | Возраст |
| `sibsp`      | int      | Количество братьев/сестёр/супругов на борту |
| `parch`      | int      | Количество родителей/детей на борту |
| `fare`       | float    | Стоимость билета |
| `embarked`   | str      | Порт посадки (`C`=Cherbourg, `Q`=Queenstown, `S`=Southampton) |
| `class`      | str      | Класс билета (`First`, `Second`, `Third`) |
| `who`        | str      | Категория: `man`, `woman` или `child` |
| `adult_male` | bool     | Является ли взрослым мужчиной |
| `deck`       | str      | Палуба |
| `embark_town`| str      | Название порта посадки |
| `alive`      | str      | Выжил (`yes`/`no`) |
| `alone`      | bool     | Путешествовал один |

**Загрузка датасета**

In [2]:
import seaborn as sns
import pandas as pd
import numpy as np


titanic_data = sns.load_dataset("titanic")
titanic_data.sample(5)

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
481,0,2,male,NaN,0,0,0.0000,S,Second,man,True,NaN,Southampton,no,True
328,1,3,female,31.0,1,1,20.5250,S,Third,woman,False,NaN,Southampton,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
796,1,1,female,49.0,0,0,25.9292,S,First,woman,False,D,Southampton,yes,True
402,0,3,female,21.0,1,0,9.8250,S,Third,woman,False,NaN,Southampton,no,False


**Задача**

Ниже описаны 10 небольших заданий, которые вам необходимо решить.

**Подсказка**:

В некоторых заданиях вам может быть полезен метод `value_counts`.

### Часть 1

Определите число пропущенных данных для каждого столбца таблицы `titanic_data`.

In [3]:
null_counts = titanic_data.isnull().sum()
print(null_counts)

survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           688
embark_town      2
alive            0
alone            0
dtype: int64


### Часть 2

Удалите все столбцы, количество пропусков в которых превышает половину количества строк в таблице.

После того, как вы удалите все столбцы, нарушающие описанное условие, удалите все строки, количество пропусков в которых превышает половину количества столбцов.

In [6]:
rows, cols = titanic_data.shape

max_missing_cols = rows / 2
cols_to_drop = titanic_data.columns[titanic_data.isnull().sum() > max_missing_cols]
titanic_clean = titanic_data.drop(columns=cols_to_drop)

new_cols = titanic_clean.shape[1]
max_missing_rows = new_cols / 2
rows_to_drop = titanic_clean[titanic_clean.isnull().sum(axis=1) > max_missing_rows].index
titanic_clean = titanic_clean.drop(index=rows_to_drop)

print(f"Удалено столбцов: {len(cols_to_drop)}")
print(f"Удалено строк: {len(rows_to_drop)}")
print(f"Оставшийся размер: {titanic_clean.shape}")

Удалено столбцов: 1
Удалено строк: 0
Оставшийся размер: (891, 14)


### Часть 3

Если вы сделали все правильно, больше всего пропусков должно остаться в столбце `"age"` - 177. Их необходимо заполнить. Заполним пропуски следующим образом:
- Если значение столбца `"who"="man"`, пропуск необходимо заполнить медианным значением известных возрастов мужчин, округленным до ближайшего целого числа;
- Если значение столбца `"who"="woman"`, пропуск необходимо заполнить медианным значением известных возрастов женщин, округленным до ближайшего целого числа;
- Если значение столбца `"who"="child"`, пропуск необходимо заполнить медианным значением известных возрастов детей, округленным до ближайшего целого числа;

In [7]:
titanic_filled = titanic_clean.copy()

median_man = titanic_filled[titanic_filled['who'] == 'man']['age'].median()
median_woman = titanic_filled[titanic_filled['who'] == 'woman']['age'].median()
median_child = titanic_filled[titanic_filled['who'] == 'child']['age'].median()

median_man = int(round(median_man))
median_woman = int(round(median_woman))
median_child = int(round(median_child))

titanic_filled.loc[(titanic_filled['age'].isnull()) & (titanic_filled['who'] == 'man'), 'age'] = median_man
titanic_filled.loc[(titanic_filled['age'].isnull()) & (titanic_filled['who'] == 'woman'), 'age'] = median_woman
titanic_filled.loc[(titanic_filled['age'].isnull()) & (titanic_filled['who'] == 'child'), 'age'] = median_child

print(f"Пропусков в age после заполнения: {titanic_filled['age'].isnull().sum()}")

Пропусков в age после заполнения: 0


### Часть 4

Удалите все строки, в которых осталось больше одного пропуска. Если вы все сделали правильно, после этого действия в таблице не должно остаться пропусков.

In [9]:
titanic_final = titanic_filled[titanic_filled.isnull().sum(axis=1) <= 1]

print(f"Всего пропусков в таблице: {titanic_final.isnull().sum().sum()}")
print(f"Размер итоговой таблицы: {titanic_final.shape}")

Всего пропусков в таблице: 0
Размер итоговой таблицы: (889, 14)


### Часть 5

Определите название города, из которого отправилось больше всего пассажиров.

In [10]:
city_counts = titanic_final['embark_town'].value_counts()
most_common_city = city_counts.idxmax()

print(f"Город с max пассажиров: {most_common_city}")
print(city_counts)

Город с max пассажиров: Southampton
embark_town
Southampton    644
Cherbourg      168
Queenstown      77
Name: count, dtype: int64


### Часть 6

Определите процент выживших пассажиров от числа пассажиров, оставшихся в таблице после очистки данных. Ответ округлите до 2 знаков после запятой.

In [11]:
survived_pct = (titanic_final['survived'].sum() / len(titanic_final)) * 100
survived_pct = round(survived_pct, 2)

print(f"Процент выживших: {survived_pct}%")

Процент выживших: 38.25%


### Часть 7

Определите число выживших пассажиров для каждого пункта отправления. В ответе должен получиться объект типа `pd.Series`, индексы которого - названия пунктов отправления, а значения - число выживших пассажиров.

In [12]:
survived_by_embark = titanic_final.groupby('embark_town')['survived'].sum()

print(survived_by_embark)

embark_town
Cherbourg       93
Queenstown      30
Southampton    217
Name: survived, dtype: int64


### Часть 8

Определите процент выживших пассажиров в каждом классе. Значения округлите до 2 знаков после запятой. В ответе должен получиться объект типа `pd.Series`, индексы которого - названия классов, а значения - процент выживших пассажиров.

In [13]:
survival_by_class = titanic_final.groupby('class')['survived'].mean() * 100
survival_by_class = survival_by_class.round(2)

print(survival_by_class)

class
First     62.62
Second    47.28
Third     24.24
Name: survived, dtype: float64


### Часть 9

Будем считать, что пассажиры, купившие билет **НЕ МЕНЕЕ** чем за $100, считаются богатыми. Определите процент выживших среди богатых пассажиров. Ответ округлите до 2 знаков после запятой. В ответе должно получиться число. 

In [14]:
rich = titanic_final[titanic_final['fare'] >= 100]
if len(rich) > 0:
    rich_survived_pct = (rich['survived'].sum() / len(rich)) * 100
else:
    rich_survived_pct = 0
rich_survived_pct = round(rich_survived_pct, 2)

print(f"Богатых пассажиров: {len(rich)}")
print(f"Процент выживших среди них: {rich_survived_pct}%")

Богатых пассажиров: 53
Процент выживших среди них: 73.58%


### Часть 10

Определите количество детей, путешествовавших в одиночку.

In [15]:
children_alone = titanic_final[(titanic_final['who'] == 'child') & (titanic_final['alone'] == True)]
count_children_alone = len(children_alone)

print(f"Детей в одиночку: {count_children_alone}")

Детей в одиночку: 6


Какие выводы вы можете сделать о выживших пассажирах Титаника? 

им сильно повезло, рад что они выжили. а вообще печально так все, столько грусти тут, столько утрат(